# Week 4: Vector Power-Up & Raster Integration

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Master vector aggregation (dissolve & groupby)
2. Understand raster data as NumPy arrays with spatial metadata
3. Compute terrain slope from DEM
4. Extract raster values into vector shapes (zonal statistics)

**Packages needed:** `geopandas`, `rioxarray`, `rasterstats`, `numpy`, `matplotlib`

> If you haven't installed them yet, run:  
> `pip install rioxarray rasterstats geopandas folium mapclassify matplotlib`

## Cell [1] — Environment Setup

Import all necessary packages and load Week 3 township data.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr
import warnings
import glob
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = ['Arial Unicode MS', 'Heiti TC', 'sans-serif']

# Load TGOS townships (try URL first, fallback to local file)
try:
    from urllib.parse import quote
    TGOS_BASE = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/'
    url = TGOS_BASE + quote('鄉(鎮、市、區)界線1140318.zip')
    townships = gpd.read_file(url, layer='TOWN_MOI_1140318')
except Exception:
    # Fallback: load from local shapefile
    shp_files = glob.glob('data/township/TOWN_MOI_*.shp')
    if not shp_files:
        shp_files = glob.glob('../week3-ARIA/data/township/TOWN_MOI_*.shp')
    townships = gpd.read_file(shp_files[0])
    print("(Using local township shapefile)")

townships = townships.to_crs(epsg=3826)
print(f"Shape: {townships.shape}")
print(f"CRS: {townships.crs}")
townships.head()

## Cell [2] — Dissolve & Groupby

**Dissolve**: Merge 368 townships → ~22 counties  
**Groupby**: Count shelters per county and sum capacity

In [ ]:
# 1. Dissolve townships → counties
counties = townships.dissolve(by='COUNTYNAME').reset_index()
print(f"Counties: {len(counties)}")
print(f"Geometry types: {counties.geometry.geom_type.unique()}")

# 2. Calculate area in km²
counties['area_km2'] = counties.geometry.area / 1e6

# 3. Top 5 largest
top5 = counties.nlargest(5, 'area_km2')[['COUNTYNAME', 'area_km2']]
print(f"\nTop 5 largest counties:")
top5

---

## 🔬 Lab 1: Vector Aggregation (20 minutes)

**Goal**: Practice dissolve & groupby before entering the raster world.

> **⚠️ 注意**：Lab 1 使用**合成資料**（synthetic data）練習 dissolve/groupby 操作。課後作業（ARIA v2.0）會使用**真實的避難收容所 CSV**，操作方式相同但資料來源不同。

### Step 1: Dissolve townships → counties

In [ ]:
# Dissolve townships → counties
counties = townships.dissolve(by='COUNTYNAME').reset_index()

# Filter Hualien
hualien = counties[counties['COUNTYNAME'] == '花蓮縣']
print(f"花蓮縣 geometry type: {hualien.geometry.geom_type.values[0]}")
print(f"花蓮縣 area: {hualien.geometry.area.values[0] / 1e6:.1f} km²")

### Step 2: Create synthetic shelter data for Hualien

In [ ]:
# Create synthetic shelter data for Hualien
np.random.seed(42)
hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].copy()

shelters_list = []
sid = 1
for _, town in hualien_towns.iterrows():
    n = np.random.randint(2, 9)
    centroid = town.geometry.centroid
    for _ in range(n):
        shelters_list.append({
            'shelter_id': sid,
            'TOWNNAME': town['TOWNNAME'],
            'capacity': np.random.randint(50, 501),
            'geometry': centroid,
        })
        sid += 1

shelters_synth = gpd.GeoDataFrame(shelters_list, crs='EPSG:3826')
print(f"Synthetic shelters: {len(shelters_synth)}")
shelters_synth.head()

### Step 3: Groupby — Statistics per town

In [ ]:
# Groupby statistics per town
town_summary = shelters_synth.groupby('TOWNNAME').agg(
    shelter_count=('shelter_id', 'count'),
    total_capacity=('capacity', 'sum'),
    avg_capacity=('capacity', 'mean'),
).reset_index()

town_summary = town_summary.sort_values('total_capacity', ascending=False)
print("Shelter statistics per town:")
town_summary

### Step 4: Merge stats back to geometry & visualize

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Merge stats back to geometry
hualien_towns_stats = hualien_towns.merge(town_summary, on='TOWNNAME', how='left')
hualien_towns_stats = hualien_towns_stats.to_crs(epsg=4326)

# Interactive map
m = hualien_towns_stats.explore(
    column='total_capacity',
    cmap='YlOrRd',
    tooltip=['TOWNNAME', 'shelter_count', 'total_capacity'],
    legend=True,
    style_kwds={'weight': 2, 'fillOpacity': 0.6},
)
m.save('outputs/lab1_hualien_capacity.html')
print("Saved outputs/lab1_hualien_capacity.html")
m

---

## Cell [3] — The Raster Glass Box: It's Just a Matrix

A raster (like a DEM) is a **NumPy array with a GPS**. Each pixel = one numeric value.

In [ ]:
# Create synthetic 100x80 DEM
from rasterio.transform import from_bounds

rows, cols = 80, 100
X, Y = np.meshgrid(np.linspace(0, 1, cols), np.linspace(0, 1, rows))
elevation = 50 + 800 * (2*X - 1)**2 + 200 * Y + 30 * np.sin(8*X) * np.cos(6*Y)
elevation = elevation.astype(np.float32)

transform = from_bounds(302000, 2638000, 303600, 2640000, cols, rows)

dem_synth = xr.DataArray(
    elevation[np.newaxis, :, :],
    dims=['band', 'y', 'x'],
    coords={
        'band': [1],
        'y': np.linspace(2640000, 2638000, rows),
        'x': np.linspace(302000, 303600, cols),
    },
)
dem_synth = dem_synth.rio.write_transform(transform)
dem_synth = dem_synth.rio.write_crs('EPSG:3826')

print(f"Shape: {dem_synth.shape}")
print(f"CRS: {dem_synth.rio.crs}")
print(f"Resolution: {dem_synth.rio.resolution()}")
print(f"Min elevation: {float(dem_synth.min()):.1f} m")
print(f"Max elevation: {float(dem_synth.max()):.1f} m")
print(f"\nTop-left 5x5 values:")
print(dem_synth.values[0, :5, :5])

## Cell [4] — Affine Transform

How does `Matrix[row, col]` become real-world `(X, Y)` coordinates?

In [ ]:
# Read affine transform
t = dem_synth.rio.transform()
print("Affine Transform Parameters:")
print(f"  a (pixel width):   {t.a:.1f} m")
print(f"  b (rotation):      {t.b}")
print(f"  c (origin X):      {t.c:.1f}")
print(f"  d (rotation):      {t.d}")
print(f"  e (pixel height):  {t.e:.1f} m")
print(f"  f (origin Y):      {t.f:.1f}")

# Manual coordinate conversion for pixel [row=5, col=10]
row, col = 5, 10
real_x = t.a * col + t.b * row + t.c
real_y = t.d * col + t.e * row + t.f
elev_val = dem_synth.values[0, row, col]

print(f"\nPixel [row={row}, col={col}]:")
print(f"  Real-world X: {real_x:.1f}")
print(f"  Real-world Y: {real_y:.1f}")
print(f"  Elevation: {elev_val:.1f} m")

## Cell [5] — DEM Visualization

Create a side-by-side plot: elevation + hillshade.

In [ ]:
from matplotlib.colors import LightSource

elev = dem_synth.values[0]
ls = LightSource(azdeg=315, altdeg=45)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elevation
im1 = axes[0].imshow(elev, cmap='terrain', origin='upper')
axes[0].set_title('Elevation', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.7, label='Elevation (m)')

# Hillshade
hs = ls.hillshade(elev, vert_exag=2)
axes[1].imshow(hs, cmap='gray', origin='upper')
axes[1].set_title('Hillshade', fontsize=13, fontweight='bold')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/cell5_dem.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved outputs/cell5_dem.png")

---

## 🔬 Lab 2: Raster Exploration on Colab (15 minutes)

### ⚡ 請選擇你的環境：

| | **Option A: Google Colab（建議）** | **Option B: 本機 fallback** |
|---|---|---|
| 適用情境 | 網路正常、Drive mount 成功 | Colab 有問題或無法連線 |
| DEM 來源 | `dem_20m_hualien.tif`（Pre-lab 已下載到 Drive） | 使用 Cell [3] 的合成 DEM (`dem_synth`) |
| 後續 Step 2-4 | 用 `dem` 變數 | 用 `dem_synth` 變數（已在前面建立） |

> **Fallback 快速切換**：如果 Colab 不通，跳過 Step 1，直接到 Step 2，把 `dem` 改成 `dem_synth` 即可繼續。
> 
> **Note**: Steps 5-6 (zonal stats & risk classification) are included for reference — they will be completed in the homework.

### Step 1: Colab Setup

In [ ]:
# Option B: Local fallback (no Colab needed)
# All packages already imported above
# DEM already created as dem_synth in Cell [3]

print("Using local environment with synthetic DEM")
dem = dem_synth  # Use synthetic DEM for Lab 2
print(f"DEM ready: {dem.shape}, CRS: {dem.rio.crs}")

### Step 2: Load DEM & Inspect

In [ ]:
# === Option B: Local fallback ===
dem = dem_synth

print(f"Shape: {dem.shape}")
print(f"CRS: {dem.rio.crs}")
print(f"Transform: {dem.rio.transform()}")
print(f"Bounds: {dem.rio.bounds()}")
print(f"Elevation range: {float(dem.min()):.1f} ~ {float(dem.max()):.1f} m")
print(f"Total pixels: {dem.shape[1] * dem.shape[2]:,}")

### Step 3: Visualize DEM

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(dem.values[0], cmap='terrain', origin='upper')
plt.colorbar(im, ax=ax, shrink=0.7, label='Elevation (m)')
ax.set_title('DEM Elevation Map', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

### Step 4: Compute Slope

In [ ]:
# Compute slope
elevation = dem.values[0]
res = abs(dem.rio.resolution()[0])  # pixel spacing
dy, dx = np.gradient(elevation, res)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

print(f"Slope min: {slope_deg.min():.1f} deg")
print(f"Slope max: {slope_deg.max():.1f} deg")
print(f"Slope mean: {slope_deg.mean():.1f} deg")
print(f"Pixels > 30 deg: {(slope_deg > 30).sum() / slope_deg.size * 100:.1f}%")

# Side-by-side visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(elevation, cmap='terrain', origin='upper')
axes[0].set_title('Elevation', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.7, label='m')

im2 = axes[1].imshow(slope_deg, cmap='YlOrRd', origin='upper')
axes[1].set_title('Slope', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.7, label='degrees')

plt.tight_layout()
plt.show()

### Step 5: Zonal Statistics

In [ ]:
import tempfile, os
import rasterio
from rasterio.transform import from_bounds
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

# Use synthetic shelters from Lab 1 (already in EPSG:3826)
# DEM is also in EPSG:3826 — CRS aligned!
print(f"Shelter CRS: {shelters_synth.crs}")
print(f"DEM CRS: {dem.rio.crs}")

# Create 200m buffers
shelter_buffers = shelters_synth.copy()
shelter_buffers['geometry'] = shelter_buffers.geometry.buffer(200)

# Save DEM and slope as temporary GeoTIFFs
dem_arr = dem.values[0]
t = dem.rio.transform()
tmp_dem = os.path.join(tempfile.gettempdir(), 'synth_dem.tif')
tmp_slope = os.path.join(tempfile.gettempdir(), 'synth_slope.tif')

profile = {
    'driver': 'GTiff', 'dtype': 'float32', 'width': dem_arr.shape[1],
    'height': dem_arr.shape[0], 'count': 1, 'crs': 'EPSG:3826',
    'transform': t, 'nodata': np.nan,
}
with rasterio.open(tmp_dem, 'w', **profile) as dst:
    dst.write(dem_arr[np.newaxis, :, :].astype(np.float32))
with rasterio.open(tmp_slope, 'w', **profile) as dst:
    dst.write(slope_deg[np.newaxis, :, :].astype(np.float32))

# Zonal stats using rasterio.mask
results = []
for _, row in shelter_buffers.iterrows():
    geom = [mapping(row.geometry)]
    try:
        with rasterio.open(tmp_dem) as src:
            masked, _ = rio_mask(src, geom, crop=True, nodata=np.nan)
        valid = masked[0][~np.isnan(masked[0])]
        with rasterio.open(tmp_slope) as src:
            s_masked, _ = rio_mask(src, geom, crop=True, nodata=np.nan)
        valid_s = s_masked[0][~np.isnan(s_masked[0])]
        results.append({
            'shelter_id': row['shelter_id'],
            'mean_elev': float(np.mean(valid)) if len(valid) > 0 else np.nan,
            'min_elev': float(np.min(valid)) if len(valid) > 0 else np.nan,
            'max_elev': float(np.max(valid)) if len(valid) > 0 else np.nan,
            'std_elev': float(np.std(valid)) if len(valid) > 0 else np.nan,
            'mean_slope': float(np.mean(valid_s)) if len(valid_s) > 0 else np.nan,
            'max_slope': float(np.max(valid_s)) if len(valid_s) > 0 else np.nan,
        })
    except Exception:
        results.append({'shelter_id': row['shelter_id'], 'mean_elev': np.nan,
                       'min_elev': np.nan, 'max_elev': np.nan, 'std_elev': np.nan,
                       'mean_slope': np.nan, 'max_slope': np.nan})

zonal_df = pd.DataFrame(results)
shelters_with_terrain = shelters_synth.merge(zonal_df, on='shelter_id')
print(f"\nZonal stats computed for {len(zonal_df)} shelters")
shelters_with_terrain[['shelter_id', 'TOWNNAME', 'mean_elev', 'max_slope']].head(10)

### Step 6: Risk Classification & Visualization

In [ ]:
# Classify terrain risk
def classify_risk(slope):
    if pd.isna(slope): return 'UNKNOWN'
    if slope > 30: return 'HIGH'
    if slope >= 20: return 'MEDIUM'
    return 'LOW'

shelters_with_terrain['risk'] = shelters_with_terrain['max_slope'].apply(classify_risk)

print("Risk distribution:")
print(shelters_with_terrain['risk'].value_counts())

# Scatter plot
colors = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#2ecc71', 'UNKNOWN': '#999'}
fig, ax = plt.subplots(figsize=(10, 6))
for risk, color in colors.items():
    subset = shelters_with_terrain[shelters_with_terrain['risk'] == risk]
    if len(subset) > 0:
        ax.scatter(subset['mean_elev'], subset['max_slope'], c=color, label=risk, s=50, alpha=0.7, edgecolors='white')

ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30 deg threshold')
ax.axhline(y=20, color='orange', linestyle='--', alpha=0.5, label='20 deg threshold')
ax.set_xlabel('Mean Elevation (m)', fontsize=12)
ax.set_ylabel('Max Slope (degrees)', fontsize=12)
ax.set_title('Shelter Terrain Risk Assessment', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/lab2_terrain_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved outputs/lab2_terrain_risk.png")

---

## Week 4 Reflection

Answer in the markdown cell below:

### My Reflection (fill in)

1. **Why is "distance to river" alone insufficient for flood risk assessment?**
   
   *Your answer:*因為河流只回答了避難所離他有多有遠 配合坡度才能證明有多危險。

2. **What causes zonal_stats to return NaN, and how do you fix it?**
   
   *Your answer:*CRS不同步導致buffer沒有疊到 所以全部是Na 。永遠把 vector 對齊 raster

3. **When combining vector and raster, should you reproject the vector or the raster? Why?**
   
   *Your answer:*永遠 reproject vector。上課提到這是ai常見的fail 會爆ram也會拖速度

4. **What was the most surprising thing you learned about raster data today?**
   
   *Your answer:*沒想過能夠利用類似數字表格貼在地圖上的方式 來表達地形的真實高度